In [ ]:
import os
import sys
import json
import logging
import requests
from datetime import datetime
from dotenv import load_dotenv

from utils.models import StateVector
from utils.ingest import (
    send_states_to_api_batch,
    send_states_to_api_single,
    API_BASE_URL,
    API_KEY,
    SOURCE_TAG,
)

load_dotenv()


# OpenSky OAuth2 + REST endpoints
OPENSKY_TOKEN_URL = (
    "https://auth.opensky-network.org/auth/realms/opensky-network/protocol/openid-connect/token"
)
OPENSKY_STATES_URL = "https://opensky-network.org/api/states/all"

# print(f"API_BASE_URL={API_BASE_URL()}")
# print(f"API_KEY: {API_KEY()}")
# print(f"SOURCE_TAG={SOURCE_TAG()}")



API_BASE_URL=https://mwswskaxy6.execute-api.us-east-1.amazonaws.com/v1
API_KEY: II797FtpqXa8zraKGoKDT3jugnQzX6VY2mYqKFCA
SOURCE_TAG=notebook-get-flights-data


In [17]:
os.environ.get("API_BASE_URL")

'https://mwswskaxy6.execute-api.us-east-1.amazonaws.com/v1/flights'

In [12]:
# logging

logger = logging.getLogger()
logger.setLevel(os.environ.get("LOG_LEVEL", "INFO"))

if not logger.handlers:
    handler = logging.StreamHandler(sys.stdout)
    formatter = logging.Formatter(
        "%(asctime)s - %(levelname)s - %(name)s - %(message)s"
    )
    handler.setFormatter(formatter)
    logger.addHandler(handler)


In [9]:
# opensky functions

def _get_opensky_credentials_from_secret():
    """
    Resolve OpenSky OAuth2 credentials from environment variables.

    Expected env vars (provided via .env):
      OPENSKY_CLIENT_ID
      OPENSKY_CLIENT_SECRET
    """
    try:
        client_id = os.environ.get("OPENSKY_CLIENT_ID")
        client_secret = os.environ.get("OPENSKY_CLIENT_SECRET")
        if not client_id or not client_secret:
            logger.error("OpenSky credentials missing in environment")
            return None, None
        return client_id, client_secret
    except Exception as e:
        logger.error(f"Error retrieving OpenSky credentials: {e}")
        return None, None


def get_opensky_access_token():
    """
    Authenticate against OpenSky OAuth2 (client_credentials) and return
    the access_token used as a Bearer token on /api/states/all.
    """
    client_id, client_secret = _get_opensky_credentials_from_secret()
    if not client_id or not client_secret:
        return None

    data = {
        "grant_type": "client_credentials",
        "client_id": client_id,
        "client_secret": client_secret,
    }

    try:
        resp = requests.post(OPENSKY_TOKEN_URL, data=data, timeout=10)
        resp.raise_for_status()
        access_token = resp.json().get("access_token")
        if not access_token:
            logger.error("No access_token in OpenSky auth response")
            return None
        logger.info("Successfully obtained OpenSky access token")
        return access_token
    except requests.RequestException as e:
        logger.error(f"Error obtaining OpenSky access token: {e}")
        return None


def get_opensky_states(access_token):
    """
    Call /api/states/all with a Bearer token and return a list of
    StateVector instances parsed from the OpenSky positional array.
    """
    headers = {"Authorization": f"Bearer {access_token}"}

    try:
        resp = requests.get(OPENSKY_STATES_URL, headers=headers, timeout=15)
        resp.raise_for_status()
        body = resp.json()
    except requests.RequestException as e:
        logger.error(f"Error calling OpenSky states API: {e}")
        return []

    raw_states = body.get("states", []) or []
    states: list[StateVector] = []

    for row in raw_states:
        try:
            state = StateVector.from_api_response(row)
            states.append(state)
        except Exception as e:
            logger.warning(f"Failed to parse state vector: {e}")

    logger.info(f"Retrieved {len(states)} state vectors from OpenSky API")
    return states


def states_to_api_payload(states: list[StateVector]) -> list[dict]:
    """Convert StateVector objects into the JSON shape the edge API expects."""
    return [s.to_api_payload() for s in states]


In [13]:
# Smoke test: one call to OpenSky, send a small batch to the API.
access_token = get_opensky_access_token()
states = get_opensky_states(access_token)
payload = states_to_api_payload(states)
print(f"states: {len(states)}, payload size: {len(payload)}")
print("sample:", json.dumps(payload[0], default=str) if payload else "<empty>")

summary = send_states_to_api_batch(payload, source=SOURCE_TAG())
summary

2026-06-11 21:43:00,525 - INFO - root - Successfully obtained OpenSky access token
2026-06-11 21:43:03,744 - INFO - root - Retrieved 8206 state vectors from OpenSky API
states: 8206, payload size: 8206
sample: {"icao24": "ab2972", "callsign": "N8180F", "origin_country": "United States", "time_position": 1781224979, "last_contact": 1781224979, "longitude": -90.2462, "latitude": 36.029, "altitude": 2225.04, "on_ground": false, "velocity": 47.77, "heading": 221.07, "vertical_rate": 2.6, "spi": false, "position_source": 0}
2026-06-11 21:43:04,780 - ERROR - utils.ingest - batch chunk rejected by API: status=403 body={'message': 'Missing Authentication Token'}
2026-06-11 21:43:05,876 - ERROR - utils.ingest - batch chunk rejected by API: status=403 body={'message': 'Missing Authentication Token'}
2026-06-11 21:43:06,903 - ERROR - utils.ingest - batch chunk rejected by API: status=403 body={'message': 'Missing Authentication Token'}
2026-06-11 21:43:07,900 - ERROR - utils.ingest - batch chunk 

{'sent': 0,
 'rejected': 8206,
 'failed_chunks': 17,
 'chunks': 17,
 'duration_ms': 17668}

In [ ]:
# Production loop: BATCH transport (recommended for high-volume feeds).
# Each iteration:
#   1. Get an OpenSky access token.
#   2. Fetch /api/states/all.
#   3. Send the whole snapshot as a single batch POST to /flights/batch.
#   4. Sleep until the next cycle.

from asyncio import sleep

total_sent = 0
total_rejected = 0
duration_seconds = 90
interval_seconds = 60
dt_start = datetime.now().isoformat()

while True:
    access_token = get_opensky_access_token()
    states = get_opensky_states(access_token)
    payload = states_to_api_payload(states)
    summary = send_states_to_api_batch(payload, source=SOURCE_TAG())
    total_sent += summary["sent"]
    total_rejected += summary["rejected"]
    logger.info(
        "cycle sent=%d rejected=%d failed_chunks=%d duration_ms=%d",
        summary["sent"], summary["rejected"], summary["failed_chunks"], summary["duration_ms"],
    )
    await sleep(interval_seconds)
    elapsed = (datetime.now() - datetime.fromisoformat(dt_start)).total_seconds()
    if elapsed > duration_seconds:
        break

logger.info("loop finished: sent=%d rejected=%d", total_sent, total_rejected)


In [ ]:
# Alternative transport: SINGLE events (POST /flights per state).
# Use this for low-volume feeds or when you need per-record response
# inspection. Higher HTTP overhead, but easier to map a single bad
# record to its failure response.

from asyncio import sleep

total_sent = 0
total_rejected = 0
total_failed = 0
duration_seconds = 90
interval_seconds = 60
dt_start = datetime.now().isoformat()

while True:
    access_token = get_opensky_access_token()
    states = get_opensky_states(access_token)
    payload = states_to_api_payload(states)
    summary = send_states_to_api_single(payload, source=SOURCE_TAG())
    total_sent += summary["sent"]
    total_rejected += summary["rejected"]
    total_failed += summary["failed"]
    logger.info(
        "cycle sent=%d rejected=%d failed=%d requests=%d duration_ms=%d",
        summary["sent"], summary["rejected"], summary["failed"],
        summary["requests"], summary["duration_ms"],
    )
    await sleep(interval_seconds)
    elapsed = (datetime.now() - datetime.fromisoformat(dt_start)).total_seconds()
    if elapsed > duration_seconds:
        break

logger.info(
    "loop finished (single): sent=%d rejected=%d failed=%d",
    total_sent, total_rejected, total_failed,
)
